# Processed E-commerce Dataset

This notebook combines three related datasets — **Orders**, **Customers**, and **Products** — into a single clean, processed dataset. It demonstrates `merge()`, `concat()`, `apply()`, and DateTime operations using Pandas, and exports the final result as a CSV file.

## 1. Importing Libraries and Loading the Datasets

In [1]:
import pandas as pd

orders = pd.read_csv("Day9_Orders.csv")
customers = pd.read_csv("Day9_Customers.csv")
products = pd.read_csv("Day9_Products.csv")

print("Orders shape:", orders.shape)
print("Customers shape:", customers.shape)
print("Products shape:", products.shape)

Orders shape: (120, 7)
Customers shape: (30, 5)
Products shape: (20, 5)


In [2]:
orders.head()

,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


In [3]:
customers.head()

,Customer_ID,Customer_Name,City,Region,Membership_Type
0,C001,Aarav Sharma,Srinagar,North,Premium
1,C002,Zoya Khan,Delhi,North,Regular
2,C003,Rohan Mehta,Mumbai,West,Premium
3,C004,Ananya Singh,Jammu,North,Regular
4,C005,Kabir Ali,Lucknow,North,New


In [4]:
products.head()

,Product_ID,Product_Name,Category,Unit_Price,Brand
0,P001,Wireless Headphones,Electronics,1499,SoundMax
1,P002,Mechanical Keyboard,Electronics,2499,KeyPro
2,P003,Wireless Mouse,Electronics,899,TechGear
3,P004,Smart Watch,Electronics,3299,FitTech
4,P005,Power Bank,Electronics,1199,VoltPlus


## 2. Merging the Datasets

`merge()` combines DataFrames based on a common key column, similar to a SQL join. Here we join **Orders** with **Customers** (on `Customer_ID`) and **Products** (on `Product_ID`) to bring all related information into a single table.

In [5]:
# Merge Orders with Customers on Customer_ID
orders_customers = pd.merge(orders, customers, on="Customer_ID", how="left")

# Merge the result with Products on Product_ID
merged_df = pd.merge(orders_customers, products, on="Product_ID", how="left")

print("Merged dataset shape:", merged_df.shape)
merged_df.head()

Merged dataset shape: (120, 15)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status,Customer_Name,City,Region,Membership_Type,Product_Name,Category,Unit_Price,Brand
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered,Harsh Vardhan,Noida,North,Premium,Cricket Bat,Sports,2499,BatPro
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered,Ishita Gupta,Bengaluru,South,Premium,Wireless Mouse,Electronics,899,TechGear
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered,Karan Joshi,Chandigarh,North,Regular,Smart Watch,Electronics,3299,FitTech
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered,Maryam Khan,Hyderabad,South,Regular,Machine Learning Basics,Books,999,AIPress
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered,Reyansh Jain,Kolkata,East,New,Coffee Maker,Home & Kitchen,3499,HomeBrew


## 3. Demonstrating `concat()`

`concat()` stacks DataFrames together, either row-wise or column-wise. Below, we split the merged dataset into two halves and use `concat()` to combine them back — demonstrating how `concat()` works for combining DataFrames.

In [6]:
# Split the merged dataset into two halves (row-wise)
half_1 = merged_df.iloc[:len(merged_df)//2]
half_2 = merged_df.iloc[len(merged_df)//2:]

print("Half 1 shape:", half_1.shape)
print("Half 2 shape:", half_2.shape)

# Combine them back together using concat()
combined_df = pd.concat([half_1, half_2], ignore_index=True)
print("Combined shape after concat():", combined_df.shape)

# Confirm it matches the original merged dataset
print("Matches original merged data:", combined_df.equals(merged_df))

Half 1 shape: (60, 15)
Half 2 shape: (60, 15)
Combined shape after concat(): (120, 15)
Matches original merged data: True


## 4. DateTime Operations

Converting `Order_Date` to a proper datetime type, then extracting useful information such as month, day, and day of the week.

In [7]:
merged_df["Order_Date"] = pd.to_datetime(merged_df["Order_Date"])

merged_df["Order_Month"] = merged_df["Order_Date"].dt.month
merged_df["Order_Month_Name"] = merged_df["Order_Date"].dt.month_name()
merged_df["Order_Day"] = merged_df["Order_Date"].dt.day
merged_df["Order_Day_of_Week"] = merged_df["Order_Date"].dt.day_name()

merged_df[["Order_ID", "Order_Date", "Order_Month_Name", "Order_Day", "Order_Day_of_Week"]].head()

,Order_ID,Order_Date,Order_Month_Name,Order_Day,Order_Day_of_Week
0,O0001,2026-02-19,February,19,Thursday
1,O0002,2026-01-25,January,25,Sunday
2,O0003,2026-02-26,February,26,Thursday
3,O0004,2026-03-04,March,4,Wednesday
4,O0005,2026-03-29,March,29,Sunday


## 5. Using `apply()` to Create and Transform Columns

`apply()` runs a custom function across rows or a column. Here we use it to calculate the total price per order line, flag high-value orders, and simplify the order status.

In [8]:
# Calculate total order value (Quantity x Unit_Price) using apply() on rows
merged_df["Total_Amount"] = merged_df.apply(lambda row: row["Quantity"] * row["Unit_Price"], axis=1)

merged_df[["Order_ID", "Quantity", "Unit_Price", "Total_Amount"]].head()

,Order_ID,Quantity,Unit_Price,Total_Amount
0,O0001,2,2499,4998
1,O0002,2,899,1798
2,O0003,1,3299,3299
3,O0004,3,999,2997
4,O0005,5,3499,17495


In [9]:
# Flag high-value orders (Total_Amount > 3000) using apply() on a single column
def flag_high_value(amount):
    return "High" if amount > 3000 else "Normal"

merged_df["Order_Value_Flag"] = merged_df["Total_Amount"].apply(flag_high_value)

merged_df[["Order_ID", "Total_Amount", "Order_Value_Flag"]].head()

,Order_ID,Total_Amount,Order_Value_Flag
0,O0001,4998,High
1,O0002,1798,Normal
2,O0003,3299,High
3,O0004,2997,Normal
4,O0005,17495,High


In [10]:
# Simplify Order_Status into a boolean 'Is_Successful' column using apply()
merged_df["Is_Successful"] = merged_df["Order_Status"].apply(lambda status: status != "Cancelled")

merged_df[["Order_ID", "Order_Status", "Is_Successful"]].head()

,Order_ID,Order_Status,Is_Successful
0,O0001,Delivered,True
1,O0002,Delivered,True
2,O0003,Delivered,True
3,O0004,Delivered,True
4,O0005,Delivered,True


## 6. Organizing the Final Processed Dataset

Selecting and reordering the most meaningful columns into a clean final DataFrame.

In [11]:
final_df = merged_df[[
    "Order_ID", "Order_Date", "Order_Month_Name", "Order_Day", "Order_Day_of_Week",
    "Customer_ID", "Customer_Name", "City", "Region", "Membership_Type",
    "Product_ID", "Product_Name", "Category", "Brand",
    "Quantity", "Unit_Price", "Total_Amount", "Order_Value_Flag",
    "Payment_Method", "Order_Status", "Is_Successful"
]]

print("Final processed dataset shape:", final_df.shape)
final_df.head(10)

Final processed dataset shape: (120, 21)


,Order_ID,Order_Date,Order_Month_Name,Order_Day,Order_Day_of_Week,Customer_ID,Customer_Name,City,Region,Membership_Type,...,Product_Name,Category,Brand,Quantity,Unit_Price,Total_Amount,Order_Value_Flag,Payment_Method,Order_Status,Is_Successful
0,O0001,2026-02-19,February,19,Thursday,C027,Harsh Vardhan,Noida,North,Premium,...,Cricket Bat,Sports,BatPro,2,2499,4998,High,Credit Card,Delivered,True
1,O0002,2026-01-25,January,25,Sunday,C006,Ishita Gupta,Bengaluru,South,Premium,...,Wireless Mouse,Electronics,TechGear,2,899,1798,Normal,Debit Card,Delivered,True
2,O0003,2026-02-26,February,26,Thursday,C015,Karan Joshi,Chandigarh,North,Regular,...,Smart Watch,Electronics,FitTech,1,3299,3299,High,Cash on Delivery,Delivered,True
3,O0004,2026-03-04,March,4,Wednesday,C024,Maryam Khan,Hyderabad,South,Regular,...,Machine Learning Basics,Books,AIPress,3,999,2997,Normal,Net Banking,Delivered,True
4,O0005,2026-03-29,March,29,Sunday,C025,Reyansh Jain,Kolkata,East,New,...,Coffee Maker,Home & Kitchen,HomeBrew,5,3499,17495,High,Credit Card,Delivered,True
5,O0006,2026-02-09,February,9,Monday,C003,Rohan Mehta,Mumbai,West,Premium,...,Football,Sports,SportZone,3,799,2397,Normal,UPI,Delivered,True
6,O0007,2026-02-10,February,10,Tuesday,C011,Vivaan Kapoor,Jaipur,North,New,...,Wireless Mouse,Electronics,TechGear,3,899,2697,Normal,UPI,Delivered,True
7,O0008,2026-03-27,March,27,Friday,C020,Priya Menon,Chennai,South,Premium,...,Power Bank,Electronics,VoltPlus,4,1199,4796,High,Debit Card,Delivered,True
8,O0009,2026-03-13,March,13,Friday,C024,Maryam Khan,Hyderabad,South,Regular,...,Coffee Maker,Home & Kitchen,HomeBrew,2,3499,6998,High,UPI,Cancelled,False
9,O0010,2026-03-05,March,5,Thursday,C026,Fatima Noor,Srinagar,North,Regular,...,Power Bank,Electronics,VoltPlus,2,1199,2398,Normal,Debit Card,Shipped,True


## 7. Quick Check for Missing Values

Confirming the merges didn't introduce any unmatched (missing) records.

In [12]:
final_df.isnull().sum()

Order_ID             0
Order_Date           0
Order_Month_Name     0
Order_Day            0
Order_Day_of_Week    0
Customer_ID          0
Customer_Name        0
City                 0
Region               0
Membership_Type      0
Product_ID           0
Product_Name         0
Category             0
Brand                0
Quantity             0
Unit_Price           0
Total_Amount         0
Order_Value_Flag     0
Payment_Method       0
Order_Status         0
Is_Successful        0
dtype: int64

## 8. Exporting the Final Processed Dataset

In [13]:
final_df.to_csv("Processed_Ecommerce_Dataset.csv", index=False)
print("Processed dataset exported as 'Processed_Ecommerce_Dataset.csv'")

Processed dataset exported as 'Processed_Ecommerce_Dataset.csv'


## 9. Observations

- The final processed dataset combines **Orders**, **Customers**, and **Products** into a single table of 120 rows, using `merge()` on `Customer_ID` and `Product_ID`.
- `concat()` was used to demonstrate splitting and recombining the dataset, confirming that row-wise concatenation reproduces the original merged data exactly.
- New columns such as `Order_Month_Name`, `Order_Day`, and `Order_Day_of_Week` were extracted from `Order_Date` using DateTime operations, enabling time-based analysis (e.g., which days or months see more orders).
- `apply()` was used to engineer new columns: `Total_Amount` (Quantity × Unit_Price), `Order_Value_Flag` (High/Normal based on order value), and `Is_Successful` (based on order status).
- No missing values were introduced during merging, confirming that all `Customer_ID` and `Product_ID` values in Orders had matching records in the Customers and Products tables.
- The final dataset is now clean, enriched, and ready for further analysis (e.g., sales trends by day of week, high-value customer segments, or category-wise performance).